# 03 — Dataset OCR: Potong Plat dan Beri Label Teks
Dataset publik hanya punya **kotak plat**, bukan **teks platnya**. Untuk melatih OCR, setiap potongan plat harus diberi label teks, misalnya `B1234XYZ`.

Alur notebook ini:
1. Memotong plat dari `data/detect` memakai kotak anotasi (lebih tepat daripada prediksi model).
2. *(Opsional)* Memotong plat dari foto lokal `data/raw/lokal` memakai `yolov8n_plate_best.pt`.
3. **Pra-label otomatis** dengan EasyOCR, supaya kamu cukup mengoreksi, bukan mengetik dari nol.
4. **Alat labeling** di Colab: cek atau koreksi teks, lalu simpan. Progres tersimpan ke Drive dan bisa dilanjutkan kapan saja.
5. Ekspor daftar label format PaddleOCR (`train_list.txt`, `val_list.txt`, `test_list.txt`).

**Target label terverifikasi:** semua potongan **test** dan **valid** (±1.100), plus **≥ 1.000** potongan **train**.
Dengan pra-label, kebanyakan potongan cukup dicek lalu klik Simpan.

**Runtime:** T4 GPU (untuk EasyOCR). Notebook ini tidak memakai PaddlePaddle.

## 1. Drive, instalasi, utilitas plat

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
ROOT = '/content/drive/MyDrive/0-KEPSTON/model_alpr'
OCR = f'{ROOT}/data/ocr'

!pip install -q -U ultralytics easyocr

In [ ]:
from pathlib import Path
import sys
CODE = Path(f'{ROOT}/code'); CODE.mkdir(parents=True, exist_ok=True)
(CODE / 'plate_utils.py').write_text('"""Utilitas plat nomor Indonesia: pembersihan, koreksi karakter per posisi, validasi pola."""\nimport re\n\nLETTER_FIX = {\'0\': \'O\', \'1\': \'I\', \'2\': \'Z\', \'4\': \'A\', \'5\': \'S\', \'6\': \'G\', \'7\': \'T\', \'8\': \'B\'}\nDIGIT_FIX = {\'O\': \'0\', \'D\': \'0\', \'Q\': \'0\', \'U\': \'0\', \'I\': \'1\', \'L\': \'1\', \'J\': \'1\',\n             \'Z\': \'2\', \'A\': \'4\', \'S\': \'5\', \'G\': \'6\', \'T\': \'7\', \'B\': \'8\'}\nPATTERN = re.compile(r\'^([A-Z]{1,2})([0-9]{1,4})([A-Z]{0,3})$\')\n\n# Kode wilayah yang umum dipakai. Dipakai sebagai penalti lunak (bukan penolakan),\n# sehingga kode yang tidak ada di daftar tetap bisa diterima bila tidak ada pilihan lain.\nREGION_CODES = set(\'A B D E F G H K L M N P R S T W Z\'.split()) | set(\n    \'AA AB AD AE AG BA BB BD BE BG BH BK BL BM BN BP DA DB DC DD DE DG DH DK DL DM DN DR DS DT \'\n    \'EA EB ED KB KH KT KU PA PB\'.split())\n\n\ndef clean(text):\n    """Huruf kapital, buang semua selain A-Z dan 0-9."""\n    return re.sub(r\'[^A-Z0-9]\', \'\', str(text).upper())\n\n\ndef fix_plate(text):\n    """Cari pembagian [huruf 1-2][angka 1-4][huruf 0-3] dengan koreksi karakter paling sedikit.\n    Mengembalikan string tanpa spasi (mis. \'B1234XYZ\') atau None bila tidak bisa dibentuk."""\n    s = clean(text)\n    if len(s) > 9 and s[-4:].isdigit():        # sisa baris masa berlaku (MM.YY) ikut terbaca\n        s = s[:-4]\n    best = None\n    for a in (1, 2):\n        for b in range(1, 5):\n            c = len(s) - a - b\n            if c < 0 or c > 3:\n                continue\n            out, cost, ok = [], 0, True\n            for i, ch in enumerate(s):\n                if i < a or i >= a + b:          # posisi huruf\n                    if ch.isalpha():\n                        out.append(ch)\n                    elif ch in LETTER_FIX:\n                        out.append(LETTER_FIX[ch]); cost += 1\n                    else:\n                        ok = False; break\n                else:                             # posisi angka\n                    if ch.isdigit():\n                        out.append(ch)\n                    elif ch in DIGIT_FIX:\n                        out.append(DIGIT_FIX[ch]); cost += 1\n                    else:\n                        ok = False; break\n            if ok:\n                if \'\'.join(out[:a]) not in REGION_CODES:\n                    cost += 2                     # kode wilayah tidak dikenal\n                key = (cost, -b)\n                if best is None or key < best[0]:\n                    best = (key, \'\'.join(out))\n    return best[1] if best else None\n\n\ndef is_valid(plate):\n    return bool(plate) and bool(PATTERN.match(plate))\n\n\ndef format_plate(plate):\n    """\'B1234XYZ\' -> \'B 1234 XYZ\'"""\n    m = PATTERN.match(plate or \'\')\n    return \' \'.join(g for g in m.groups() if g) if m else (plate or \'\')\n\n\ndef levenshtein(a, b):\n    a, b = a or \'\', b or \'\'\n    prev = list(range(len(b) + 1))\n    for i, ca in enumerate(a, 1):\n        cur = [i]\n        for j, cb in enumerate(b, 1):\n            cur.append(min(prev[j] + 1, cur[j - 1] + 1, prev[j - 1] + (ca != cb)))\n        prev = cur\n    return prev[-1]\n')
sys.path.insert(0, str(CODE))
import importlib, plate_utils; importlib.reload(plate_utils)
from plate_utils import clean, fix_plate, is_valid, format_plate, levenshtein
print(fix_plate('b 1234 xyz'), fix_plate('B1Z34XY2'), fix_plate('BI234XYZ 05.28'))

## 2. Potong plat dari dataset deteksi
Margin kecil ditambahkan di sekeliling kotak (4% lebar, 10% tinggi). Margin yang **sama** dipakai lagi di pipeline akhir (notebook 05).
Potongan yang terlalu kecil (tinggi < 12 px atau lebar < 30 px) dilewati karena teksnya tidak terbaca.

Sel ini **tidak menimpa** label yang sudah dibuat: kalau `labels.csv` sudah ada, sel dilewati.

In [ ]:
import cv2, pandas as pd
from pathlib import Path

CROP_MARGIN_X, CROP_MARGIN_Y = 0.04, 0.10
DET = Path(f'{ROOT}/data/detect')
CSV = Path(f'{OCR}/labels.csv')

def crop_plate(img, x1, y1, x2, y2):
    h, w = img.shape[:2]
    bw, bh = x2 - x1, y2 - y1
    x1 = int(max(0, x1 - bw * CROP_MARGIN_X)); x2 = int(min(w, x2 + bw * CROP_MARGIN_X))
    y1 = int(max(0, y1 - bh * CROP_MARGIN_Y)); y2 = int(min(h, y2 + bh * CROP_MARGIN_Y))
    return img[y1:y2, x1:x2]

if CSV.exists():
    df = pd.read_csv(CSV, dtype=str).fillna('')
    print('labels.csv sudah ada, dilewati. Jumlah baris:', len(df))
else:
    rows = []
    for split in ['train', 'valid', 'test']:
        out_dir = Path(OCR) / 'images' / split
        out_dir.mkdir(parents=True, exist_ok=True)
        for img_path in sorted((DET / split / 'images').glob('*')):
            lbl = DET / split / 'labels' / (img_path.stem + '.txt')
            if not lbl.exists() or not lbl.read_text().strip():
                continue
            img = cv2.imread(str(img_path))
            if img is None:
                continue
            H, W = img.shape[:2]
            for k, ln in enumerate(lbl.read_text().splitlines()):
                _, cx, cy, bw, bh = map(float, ln.split())
                x1, y1 = (cx - bw / 2) * W, (cy - bh / 2) * H
                x2, y2 = (cx + bw / 2) * W, (cy + bh / 2) * H
                crop = crop_plate(img, x1, y1, x2, y2)
                if crop.shape[0] < 12 or crop.shape[1] < 30:
                    continue
                name = f'{img_path.stem}_{k}.jpg'
                cv2.imwrite(str(out_dir / name), crop, [cv2.IMWRITE_JPEG_QUALITY, 95])
                rows.append({'crop': f'images/{split}/{name}', 'split': split,
                             'source': f'{split}/images/{img_path.name}',
                             'prelabel': '', 'prescore': '', 'text': '', 'status': ''})
    df = pd.DataFrame(rows)
    df.to_csv(CSV, index=False)
    print(df.split.value_counts())

## 3. (Opsional) Potong plat dari foto lokal gerbang FT UNTIRTA
Simpan foto lokal di `data/raw/lokal/`. Plat dideteksi dengan `yolov8n_plate_best.pt` dan masuk split **`test_lokal`**,
yaitu data uji akhir yang **tidak pernah** dipakai training. Sel ini aman dijalankan ulang setelah menambah foto baru.

In [ ]:
from ultralytics import YOLO

LOKAL = Path(f'{ROOT}/data/raw/lokal')
photos = [p for p in LOKAL.rglob('*') if p.suffix.lower() in {'.jpg', '.jpeg', '.png'}]
print('Foto lokal:', len(photos))
if photos:
    det = YOLO(f'{ROOT}/models/yolov8n_plate_best.pt')
    out_dir = Path(OCR) / 'images' / 'test_lokal'; out_dir.mkdir(parents=True, exist_ok=True)
    known = set(df['crop'])
    new_rows = []
    for p in photos:
        img = cv2.imread(str(p))
        if img is None:
            continue
        res = det.predict(img, imgsz=640, conf=0.5, verbose=False)[0]
        for k, (x1, y1, x2, y2) in enumerate(res.boxes.xyxy.cpu().numpy()):
            crop = crop_plate(img, x1, y1, x2, y2)
            name = f'{p.stem}_{k}.jpg'
            rel = f'images/test_lokal/{name}'
            if rel in known or crop.shape[0] < 12 or crop.shape[1] < 30:
                continue
            cv2.imwrite(str(out_dir / name), crop, [cv2.IMWRITE_JPEG_QUALITY, 95])
            new_rows.append({'crop': rel, 'split': 'test_lokal', 'source': f'lokal/{p.relative_to(LOKAL)}',
                             'prelabel': '', 'prescore': '', 'text': '', 'status': ''})
    if new_rows:
        df = pd.concat([df, pd.DataFrame(new_rows)], ignore_index=True)
        df.to_csv(CSV, index=False)
    print('Potongan lokal baru:', len(new_rows))

## 4. Pra-label otomatis dengan EasyOCR
EasyOCR membaca setiap potongan, lalu hasilnya dirapikan dengan `fix_plate`. Baris kecil masa berlaku (MM.YY) dibuang
dengan hanya mengambil teks berukuran besar (baris nomor utama).
Hanya baris yang belum punya pra-label yang diproses, jadi aman dijalankan ulang. Progres disimpan setiap 200 potongan.

In [ ]:
import easyocr
reader = easyocr.Reader(['en'], gpu=True)
ALLOW = '0123456789ABCDEFGHIJKLMNOPQRSTUVWXYZ'

def easy_read(path):
    res = reader.readtext(str(path), allowlist=ALLOW, detail=1, paragraph=False)
    if not res:
        return '', 0.0
    hs = [max(p[1] for p in b) - min(p[1] for p in b) for b, _, _ in res]
    top = max(hs)
    main = [(min(p[0] for p in b), t, c) for (b, t, c), h in zip(res, hs) if h >= 0.6 * top]
    main.sort()
    text = ''.join(t for _, t, _ in main)
    conf = sum(c for _, _, c in main) / len(main)
    return text, conf

todo = df.index[df['prelabel'] == ''].tolist()
print('Perlu pra-label:', len(todo))
for n_done, i in enumerate(todo, 1):
    raw, conf = easy_read(Path(OCR) / df.at[i, 'crop'])
    df.at[i, 'prelabel'] = fix_plate(raw) or clean(raw) or '?'
    df.at[i, 'prescore'] = f'{conf:.2f}'
    if n_done % 200 == 0:
        df.to_csv(CSV, index=False); print(n_done, 'selesai')
df.to_csv(CSV, index=False)
print('Pra-label selesai. Pra-label yang sudah valid:', df['prelabel'].map(is_valid).sum(), 'dari', len(df))

## 5. Alat labeling
- Gambar potongan plat tampil di atas, kotak teks berisi pra-label.
- **Benar** → klik **Simpan ✓**. **Salah** → perbaiki teksnya (tanpa spasi pun boleh), lalu **Simpan ✓**.
- **Bukan plat / tidak terbaca** → klik tombol merah (data dikeluarkan dari dataset).
- Urutan: `test_lokal` → `test` → `valid` → `train`. Setiap klik langsung tersimpan ke `labels.csv` di Drive.

Aturan label: hanya nomor utama (kode wilayah + angka + seri), **tanpa** baris masa berlaku.
Teks yang tidak cocok pola plat Indonesia ditolak dengan peringatan merah.

In [ ]:
import ipywidgets as w
from IPython.display import display
from google.colab import output
output.enable_custom_widget_manager()

df = pd.read_csv(CSV, dtype=str).fillna('')
ORDER = {'test_lokal': 0, 'test': 1, 'valid': 2, 'train': 3}
queue = sorted(df.index[df['status'] == ''].tolist(), key=lambda i: (ORDER.get(df.at[i, 'split'], 9), i))
state = {'pos': 0}

img_w = w.Image(format='jpg', layout=w.Layout(max_width='520px'))
txt = w.Text(description='Plat:', layout=w.Layout(width='340px'))
info, warn = w.HTML(), w.HTML()
b_ok = w.Button(description='Simpan ✓', button_style='success')
b_bad = w.Button(description='Bukan plat / tidak terbaca', button_style='danger', layout=w.Layout(width='240px'))
b_skip = w.Button(description='Lewati dulu')

def progress():
    s = df['status']
    by = df[s == 'ok'].groupby('split').size().to_dict()
    return f"Terverifikasi: <b>{(s == 'ok').sum()}</b> | dibuang: {(s == 'bad').sum()} | per split: {by}"

def show():
    warn.value = ''
    if state['pos'] >= len(queue):
        info.value = '<b>Semua potongan sudah dilabeli.</b> ' + progress(); return
    i = queue[state['pos']]
    img_w.value = open(Path(OCR) / df.at[i, 'crop'], 'rb').read()
    txt.value = df.at[i, 'text'] or (df.at[i, 'prelabel'] if df.at[i, 'prelabel'] != '?' else '')
    info.value = (f"Split: <b>{df.at[i, 'split']}</b> | pra-label: {df.at[i, 'prelabel']} "
                  f"(skor {df.at[i, 'prescore']})<br>{progress()}")

def commit(status, text=''):
    i = queue[state['pos']]
    df.at[i, 'status'], df.at[i, 'text'] = status, text
    df.to_csv(CSV, index=False)
    state['pos'] += 1; show()

def on_ok(_):
    p = fix_plate(txt.value)
    if is_valid(p):
        commit('ok', p)
    else:
        warn.value = '<span style="color:#c00"><b>Format tidak sesuai pola plat Indonesia.</b> Periksa lagi.</span>'

b_ok.on_click(on_ok)
b_bad.on_click(lambda _: commit('bad'))
b_skip.on_click(lambda _: (state.update(pos=state['pos'] + 1), show()))
try:
    txt.on_submit(on_ok)          # tekan Enter = Simpan
except Exception:
    pass

display(w.VBox([img_w, txt, w.HBox([b_ok, b_bad, b_skip]), warn, info]))
show()

## 6. Ekspor daftar label untuk PaddleOCR
Format satu baris: `images/train/xxx.jpg<TAB>B1234XYZ`. Pembagian mengikuti split deteksi, jadi foto yang sama tidak bocor antar-split.

In [ ]:
df = pd.read_csv(CSV, dtype=str).fillna('')
ok = df[df['status'] == 'ok']
NAMES = {'train': 'train_list.txt', 'valid': 'val_list.txt', 'test': 'test_list.txt', 'test_lokal': 'test_lokal_list.txt'}
for split, fname in NAMES.items():
    part = ok[ok['split'] == split]
    if len(part):
        (Path(OCR) / fname).write_text('\n'.join(f"{c}\t{t}" for c, t in zip(part['crop'], part['text'])) + '\n')
    print(f'{fname:22s} {len(part):5d} label')

print('\nPanjang teks:', ok['text'].str.len().value_counts().sort_index().to_dict())
print('Karakter terpakai:', ''.join(sorted(set(''.join(ok['text'])))))

---
### Checklist 03
- [ ] `test_list.txt` dan `val_list.txt` mencakup semua potongan test dan valid
- [ ] `train_list.txt` berisi ≥ 1.000 label
- [ ] *(Bila ada foto lokal)* `test_lokal_list.txt` terisi

**Berikutnya:** `04_paddleocr.ipynb` — **buka di sesi Colab baru** (Runtime → Disconnect and delete runtime dulu).